<a href="https://colab.research.google.com/github/TeresaValero/Proyectos/blob/main/Teresa_Valero_Examen_m%C3%B3dulo_3_Predicci%C3%B3n_suicidios_con_RNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROBLEMA DE CLASIFICACIÓN CON RNA Y KERAS**

In [1]:
import os
import pandas as pd
import numpy as np
import random
from numpy.random import seed
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


import tensorflow as tf
from keras.models import Sequential
from keras.layers import LeakyReLU
from keras.layers import Dense, Input

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


###**1. Se carga la ruta**

In [4]:
# Se carga la ruta donde se encuentra la base de datos.
os.chdir(r"/content/")
df= pd.read_csv("/content/DatosFinales.csv",sep=",",encoding='latin-1')
#df = pd.read_csv(r"datos/DatosFinales.csv", sep=",", header=None)#Si la primera fila no es de encabezados

### **2. Se presentan los datos**

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27901 entries, 0 to 27900
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dm_Suicidal_thoughts_Yes  27901 non-null  int64  
 1   Age                       27901 non-null  float64
 2   CGPA                      27901 non-null  float64
 3   Sleep_hrs                 27901 non-null  float64
 4   Academic_Pressure         27901 non-null  float64
 5   Study_Satisfaction        27901 non-null  float64
 6   Financial_Stress          27901 non-null  float64
 7   dm_Depression_1.0         27901 non-null  int64  
 8   dm_FHMI_Yes               27901 non-null  int64  
 9   Study_hrs                 27901 non-null  float64
 10  Diet_J                    27901 non-null  float64
 11  Mill_hbs                  27901 non-null  float64
 12  Num_NivEst                27901 non-null  float64
 13  Male                      27901 non-null  int64  
 14  Studen

In [6]:
df.head()

,dm_Suicidal_thoughts_Yes,Age,CGPA,Sleep_hrs,Academic_Pressure,Study_Satisfaction,Financial_Stress,dm_Depression_1.0,dm_FHMI_Yes,Study_hrs,...,dm_area_Ade,dm_area_Arquitectura,dm_area_Arte,dm_area_Ciencias,dm_area_Derecho,dm_area_Educacion,dm_area_Humanidades,dm_area_Medicina,dm_area_Tech,dm_area_Turismo
0,1,33.0,8.97,6.0,5.0,2.0,1.0,1,0,3.0,...,0,0,0,0,0,0,0,1,0,0
1,0,24.0,5.90,6.0,2.0,5.0,2.0,0,1,3.0,...,0,0,0,1,0,0,0,0,0,0
2,0,31.0,7.03,5.0,3.0,5.0,1.0,0,1,9.0,...,0,0,1,0,0,0,0,0,0,0
3,1,28.0,5.59,8.0,3.0,2.0,5.0,1,1,4.0,...,0,0,0,0,0,0,0,0,1,0
4,1,25.0,8.13,6.0,4.0,3.0,1.0,0,0,1.0,...,0,0,0,0,0,0,0,0,1,0


Significado de las columnas:

0. dm_Suicidal_thoughts_Yes
1. Age
2. CGPA
3. Sleep_hrs
4. Academic_Pressure
5. Study_Satisfaction
6. Financial_Stress
7. dm_Depression_1.0
8. dm_FHMI_Yes
9. Study_hrs
10. Diet_J
11. Mill_hbs
12. Num_NivEst
13. Male
14. Student
15. dm_area_Ade
16. dm_area_Arquitectura
17. dm_area_Arte
18. dm_area_Ciencias
19. dm_area_Derecho
20. dm_area_Educacion
21. dm_area_Humanidades
22. dm_area_Medicina
23. dm_area_Tech
24. dm_area_Turismo

### **3. Establecer variable dependiente e independientes**

In [7]:
# Dividimos los datos en X e y La variable dependiente está en la columna 0 y las independientes a partir de la columna 1 a la 25
X = df.iloc[:,1:25]
y = df.iloc[:,0]

### **4. Dividir datos en train y test**

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
                                        X,
                                        y,
                                        train_size   = 0.7,
                                        random_state = 123,
                                        shuffle      = True
                                    )

### **5. Estandarizamos los datos**

In [9]:
Columnas_a_Estandarizar = ['Age', 'Sleep_hrs', 'CGPA', 'Study_hrs']
print(X[Columnas_a_Estandarizar].head())


    Age  Sleep_hrs  CGPA  Study_hrs
0  33.0        6.0  8.97        3.0
1  24.0        6.0  5.90        3.0
2  31.0        5.0  7.03        9.0
3  28.0        8.0  5.59        4.0
4  25.0        6.0  8.13        1.0


In [10]:
# Instanciar el escalador
scaler = StandardScaler()

# Estandarizar solo las columnas seleccionadas
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Estandarizar las columnas numéricas no dummie o categóricas
X_train_scaled[Columnas_a_Estandarizar] = scaler.fit_transform(X_train[Columnas_a_Estandarizar])
X_test_scaled[Columnas_a_Estandarizar] = scaler.transform(X_test[Columnas_a_Estandarizar])

# Mostrar los datos transformados
print(X_train_scaled[Columnas_a_Estandarizar].head())

            Age  Sleep_hrs      CGPA  Study_hrs
17708 -0.580172  -1.181009 -0.455539   0.767903
10383  0.437427  -1.181009 -1.410423  -0.579961
5974  -0.376652   0.704233  1.447409   0.767903
15446 -0.783691   1.332647  0.287907   1.307048
24737  1.251505  -1.181009 -0.032661  -1.927824


### **6. Verificar las formas de los datos**

In [11]:
X_train = np.array(X_train)
y_train = np.array(y_train)

# Esto debería mostrar (n_samples, 24) para 24 características
print(X_train.shape)

# Esto debería mostrar (n_samples, 1) para clasificación binaria
print(y_train.shape)

(19530, 24)
(19530,)


### Se hace una transformación para que la variable dependiente tome la forma correcta

In [12]:
# Convertir a forma (n_samples, 1)

y_train = y_train.reshape(-1, 1)
y_test = y_test.values.reshape(-1, 1)

### **7. Se planta una semilla para obtener el mismo resultado**

In [13]:
# Se planta una semilla para numpy
np.random.seed(1)
# Semilla para random (para el generador de números aleatorios de Python)
random.seed(1)
# Semilla para TensorFlow
tf.random.set_seed(1)

### **8. Definimos el modelo secuencial**

In [14]:
#Modelo Final
model = Sequential()
model.add(Dense(12, input_shape=(24,))) #Capa de entrada con 12 neuronas pero un Input_shape de 24 de acuerdo al número de regresores
model.add(LeakyReLU(alpha=0.01))# Capa oculta con LeakyReLU
model.add(Dense(8))
model.add(LeakyReLU(alpha=0.01))# Capa oculta con LeakyReLU
model.add(Dense(1, activation='sigmoid'))# Capa de salida

#Modelo 1
#model = Sequential()
#model.add(Input(shape=(24,)))  # La entrada tiene 24 características
#model.add(Dense(12, activation='relu'))  # Capa oculta con 12 neuronas y activación ReLU
#model.add(Dense(8, activation='relu'))   # Capa oculta con 8 neuronas y activación ReLU
#model.add(Dense(1, activation='sigmoid')) # Capa de salida con 1 neurona y activación sigmoide

#Nota: Para dejar el código más limpio, la confuguración de los otros modelos probados se guardado en un fichero anexo para que quede constancia.

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


### **9. Compilación del modelo**

In [15]:
# Para definir la función de pérdida que vamos a minimizar usaremos Binary Cross Entropy que es la recomendada para problemas binarios de clasificación.
# Como métrica (al ser clasificación) se usará la precisión.
# Como optimizador, se usará el algoritmo "adam".

#Modelo 1
#model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              #loss=tf.keras.losses.BinaryCrossentropy(from_logits=False,
                                                      #label_smoothing=0.0,
                                                      #axis=-1,
                                                      #reduction="sum_over_batch_size",
                                                      #name="binary_crossentropy"),
              #metrics=['accuracy'])

# from_logits=False se usa cuando valores de predicción sean [0,1]


#Modelo Final

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Define el callback para reducir la tasa de aprendizaje
reduce_lr = ReduceLROnPlateau(monitor='val_loss',  # Monitorea la pérdida de validación
                              factor=0.5,         # Reduce la tasa de aprendizaje en un 50%
                              patience=3,         # Espera 3 épocas sin mejora antes de reducir la tasa
                              min_lr=0.0001)      # No deja que la tasa de aprendizaje baje de 0.0001

# Define el callback para early stopping
early_stopping = EarlyStopping(monitor='val_loss',  # Monitorea la pérdida de validación
                               patience=10,         # Espera 10 épocas sin mejora antes de detenerse
                               restore_best_weights=True)  # Restaura los mejores pesos al final

# Compilación del modelo
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False,
                                                      label_smoothing=0.0,
                                                      axis=-1,
                                                      reduction="sum",
                                                      name="binary_crossentropy"),
              metrics=['accuracy', 'AUC', 'Precision', 'Recall'])

### **10. Entrenamiento del modelo**

In [16]:
# Probar el entrenamiento
#Modelo 1
#model.fit(X_train, y_train, epochs=150, batch_size=20)

# Modelo final

# Entrenamiento del modelo con callbacks y batch_size=32
model.fit(X_train, y_train,
          epochs=150,
          batch_size=32,
          validation_data=(X_test, y_test),  # Proporcionamos datos de validación para monitorear 'val_loss'
          callbacks=[early_stopping, reduce_lr])  # Aquí se incluye ambos callbacks: EarlyStopping y ReduceLROnPlateau


Epoch 1/150
611/611 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - AUC: 0.4986 - Precision: 0.5938 - Recall: 0.5948 - accuracy: 0.5241 - loss: 2004.2788 - val_AUC: 0.5789 - val_Precision: 0.6451 - val_Recall: 0.9937 - val_accuracy: 0.6450 - val_loss: 27.5467 - learning_rate: 0.0010
Epoch 2/150
611/611 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - AUC: 0.6037 - Precision: 0.6725 - Recall: 0.8008 - accuracy: 0.6281 - loss: 25.1330 - val_AUC: 0.7597 - val_Precision: 0.7566 - val_Recall: 0.8580 - val_accuracy: 0.7317 - val_loss: 17.8776 - learning_rate: 0.0010
Epoch 3/150
611/611 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - AUC: 0.6674 - Precision: 0.7189 - Recall: 0.7998 - accuracy: 0.6761 - loss: 23.8574 - val_AUC: 0.7713 - val_Precision: 0.7770 - val_Recall: 0.8422 - val_accuracy: 0.7435 - val_loss: 17.2414 - learning_rate: 0.0010
Epoch 4/150
611/611 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - AUC: 0.6938 - Precision: 0.7396 - Recall: 0.7985 - accuracy: 0.6953 - loss: 23.1486 - val_AUC: 0.7647 - val_Precision: 0.7657 - val_Reca

### **11. Evaluación del modelo**

In [17]:
# Evaluar el modelo y obtener todas las métricas
results = model.evaluate(X_train, y_train, verbose=1)
# Capturamos los resultados en variables
loss, accuracy, auc, precision, recall = results
print(f'Accuracy: {accuracy*100:.2f}')
print(f'Loss: {loss:.2f}')
print(f'AUC: {auc:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')

611/611 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7815 - Precision: 0.8484 - Recall: 0.7947 - accuracy: 0.7805 - loss: 16.3463
Accuracy: 78.19
Loss: 16.33
AUC: 0.78
Precision: 0.85
Recall: 0.79


### **12. Predicciones**

In [18]:
predicciones = model.predict(X_test)

# La función sigmoide nos devueve los resultados en formato probabilidad.
# Convertimos los mismos a casos, tomando como umbral 0.5
y_pred = (predicciones > 0.5).astype(int)
y_pred

262/262 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


array([[1],
       [1],
       [0],
       ...,
       [0],
       [0],
       [0]])

### **13. Matriz de confusión**

In [19]:
confusion_matrix(y_test, y_pred)

array([[2276,  721],
       [1144, 4230]])

### **14. Otras métricas**

In [20]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.76      0.71      2997
           1       0.85      0.79      0.82      5374

    accuracy                           0.78      8371
   macro avg       0.76      0.77      0.76      8371
weighted avg       0.79      0.78      0.78      8371



### **15. Comparación de modelos**

Modelo 1:
Accuracy: 78.41

            ([[2283,  714],
            [1145, 4229]])

Modelo final:
Accuracy: 78.32

            ([[2293,  704],
            [1155, 4219]])